# Cloud Data Infrastructure - Practise work

### Osiris Yetna - Matthieu Kaeppelin

<a id="setup"></a>
# <p style="background-color: #ff6200; font-family:calibri; color:white; font-size:140%; font-family:Verdana; text-align:center; border-radius:15px 50px;">Chapter 2 | Data Model Denormalization</p>

<a id="libraries"></a>
# <b><span style='color:#fcc36d'>2.3 |</span><span style='color:#ff6200'> Relational to JSON Documents</span></b>

### 2.3.1 - Provide a JSON document example on the Product collection and merge “Categories” and “supplier”

In [1]:
json_doc_exemple = {
                    "IDP": 1050,
                    "name": "Iphone 15 Black",
                    "brand": "Apple",
                    "description": "Latest model...",
                    "image_url": "https://m.media-amazon.com/images/I/61eEYLATF9L.jpg",
                    
                    "price": {
                        "amount": 999.99,
                        "currency": "USD",
                        "VAT": 0.20
                    },

                    "supplier": {
                        "IDS": 42,
                        "name": "Apple Inc.",
                        "SIRET": "321 227 852 00118",
                        "headOffice": "Paris",
                        "Revenue": 300_000_000
                    },

                    "categories": [
                        { "title": "Telephony" },
                        { "title": "High-Tech" }
                    ]
                    }

### 2.3.2 Produce the corresponding JSON Schema

In [2]:
product_json_schema = {
  "$schema": "http://json-schema.org/draft-04/schema#",
  "type": "object",

  "required": [
    "IDP",
    "name",
    "brand",
    "description",
    "image_url",
    "price",
    "supplier",
    "categories"
  ],

  "properties": {
    "IDP": { "type": "integer" },
    "name": { "type": "string" },
    "brand": { "type": "string" },
    "description": { "type": "string" },
    "image_url": { "type": "string" },

    "price": {
      "type": "object",
      "required": ["amount", "currency", "VAT"],
      "properties": {
        "amount": { "type": "number" }, # in Python : float --> JSON : number 
        "currency": { "type": "string" },
        "VAT": { "type": "number" }
      }
    },

    "supplier": {
      "type": "object",
      "required": ["IDS", "name", "SIRET", "headOffice", "Revenue"],
      "properties": {
        "IDS": { "type": "integer" },
        "name": { "type": "string" },
        "SIRET": { "type": "string" },
        "headOffice": { "type": "string" },
        "Revenue": { "type": "integer" }
      }
    },

    "categories": {
      "type": "array",
      "items": {
        "type": "object",
        "required": ["title"],
        "properties": {
          "title": { "type": "string" }
        }
      }
    }

  }
}


<a id="libraries"></a>
# <b><span style='color:#fcc36d'>2.4 |</span><span style='color:#ff6200'> Denormalization </span></b>

## 2.4 Produce the JSON Schemas for the corresponding denormalizations of those DB signatures:

#### 2.4.0 Schema blocs

In [3]:
# Category
schema_category = {
    "type": "object",
    "properties": {"title": {"type": "string"}},
    "required": ["title"]
}

# Supplier
schema_supplier = {
    "type": "object",
    "properties": {
        "IDS": {"type": "integer"},
        "name": {"type": "string"},
        "SIRET": {"type": "string"},
        "headOffice": {"type": "string"},
        "Revenue": {"type": "integer"}
    },
    "required": ["IDS", "name", "SIRET", "headOffice", "Revenue"]
}

# Price
schema_price = {
    "type": "object",
    "properties": {
        "amount": {"type": "number"},
        "currency": {"type": "string"},
        "VAT": {"type": "number"}
    },
    "required": ["amount", "currency", "VAT"]
}

# Product (Categories + Suppliers embedded)
product_embedded_schema = {
    "type": "object",
    "properties": {
        "IDP": {"type": "integer"},
        "name": {"type": "string"},
        "brand": {"type": "string"},
        "description": {"type": "string"},
        "image_url": {"type": "string"},
        "price": schema_price,
        # Nesting [Cat]
        "categories": {"type": "array", "items": schema_category}, 
        # Nesting Supp
        "supplier": schema_supplier                               
    },
    "required": ["IDP", "name", "brand", "description", "image_url", "price", "categories", "supplier"]
}

# Warehouse 
warehouse_schema = {
    "type": "object", 
    "properties": {
        "IDW": {"type": "integer"}, 
        "address": {"type": "string"},
        "capacity": {"type": "integer"} 
    },
    "required": ["IDW", "address", "capacity"]
}

# Stock
stock_schema = {
    "type": "object", 
    "properties": {
        "IDP": {"type": "integer"}, 
        "IDW": {"type": "integer"}, 
        "quantity": {"type": "integer"},
        "location": {"type": "string"}
    },
    "required": ["IDP", "IDW", "quantity", "location"]    
}

# Order Line
order_line_schema = {
    "type": "object", 
    "properties": {
        "IDP": {"type": "integer"},
        "IDC": {"type": "integer"},
        "date": {"type": "string"},
        "quantity": {"type": "integer"},
        "deliveryDate": {"type": "string"},
        "comment": {"type": "string"},
        "grade": {"type": "integer"}
    },
    "required": ["IDP", "IDC", "quantity", "date", "deliveryDate", "comment", "grade"]
}

# Client
client_schema = {
    "type": "object", 
    "properties": {
        "IDC": {"type": "integer"},
        "ln": {"type": "string"},
        "fn": {"type": "string"},   
        "address": {"type": "string"},
        "nationality": {"type": "string"},
        "birthDate": {"type": "string"},
        "email": {"type": "string"}
    },
    "required": ["IDC", "ln", "fn", "address", "nationality", "birthDate", "email"]
}

####  DB1: Prod{[Cat],Supp}, St, Wa, OL, Cl

In [4]:
db1_schemas = {
    "Product": {
        "type": "object",
        "properties": {
            "IDP": {"type": "integer"},
            "name": {"type": "string"},
            "brand": {"type": "string"},
            "description": {"type": "string"}, 
            "image_url": {"type": "string"},   
            "price": schema_price,
            
            # Nesting: Array of Categories
            "categories": {"type": "array", "items": schema_category},
            
            # Nesting: Supplier Object
            "supplier": schema_supplier 
        },
        "required": ["IDP", "name", "brand", "description", "image_url", "price", "categories", "supplier"]
    },
    
    "Stock": stock_schema,
    "Warehouse": warehouse_schema,
    "OrderLine": order_line_schema,
    "Client": client_schema
}

### DB2: Prod{[Cat],Supp, [St]}, Wa, OL, Cl

In [5]:
db2_schemas = {
    "Product": {
        "type": "object",
        "properties": {
            "IDP": {"type": "integer"},
            "name": {"type": "string"},
            "brand": {"type": "string"},
            "description": {"type": "string"},
            "image_url": {"type": "string"},
            "price": schema_price,
            
            # Nesting: Array of Categories
            "categories": {
                "type": "array",
                "items": schema_category
                },
            
            # Nesting: Supplier Object
            "supplier": schema_supplier,
            
            # Nesting: Array of Stocks
            "stocks": {
                "type": "array",
                "items": stock_schema
            }
        },
        "required": ["IDP", "name", "brand", "description", "image_url", "price", "categories", "supplier", "stocks"]
    },
    "Warehouse": warehouse_schema,
    "OrderLine": order_line_schema,
    "Client": client_schema
}

### DB3: St{Prod{[Cat],Supp}}, Wa, OL, Cl

In [6]:
db3_schemas = {
    "Stock": {
        "type": "object",
        "properties": {
            "IDW": {"type": "integer"},
            "quantity": {"type": "integer"},
            "location": {"type": "string"},
            
            # Nesting: Prod{[Cat],Supp}
            "product": product_embedded_schema
        },
        "required": ["IDW", "quantity", "location", "product"]
    },
    
    "Warehouse": warehouse_schema,
    "OrderLine": order_line_schema,
    "Client": client_schema
}

###  DB4: St, Wa, OL{Prod{[Cat],Supp}}, Cl

In [7]:
db4_schemas = {
    "OrderLine": {
        "type": "object",
        "properties": {
            "IDC": {"type": "integer"},
            "date": {"type": "string"},
            "quantity": {"type": "integer"},
            "deliveryDate": {"type": "string"},
            "comment": {"type": "string"},
            "grade": {"type": "integer"},
            
            # Nesting: Product + Categories + Supplier
            "product": product_embedded_schema
        },
        "required": ["IDC", "date", "quantity", "grade", "product"]
    },
    "Stock": stock_schema,
    "Warehouse": warehouse_schema,
    "Client": client_schema
}

### DB5: Prod{[Cat],Supp, [OL]}, St, Wa, Cl

In [8]:
db5_schemas = {
    "Product": {
        "type": "object",
        "properties": {
            "IDP": {"type": "integer"},
            "name": {"type": "string"},
            "brand": {"type": "string"},
            "description": {"type": "string"},
            "image_url": {"type": "string"},
            "price": schema_price,
            
            # Nesting: Array of Categories
            "categories": {"type": "array", "items": schema_category},
            
            # Nesting: Supplier Object
            "supplier": schema_supplier,
            
            # Nesting: Array of OrderLines [OL]
            "order_lines": {
                "type": "array",
                "items": order_line_schema
            }
        },
        "required": ["IDP", "name", "brand", "description", "image_url", "price", "categories", "supplier", "order_lines"]
    },
    "Stock": stock_schema,
    "Warehouse": warehouse_schema,
    "Client": client_schema
}

<a id="libraries"></a>
# <b><span style='color:#fcc36d'>2.5 |</span><span style='color:#ff6200'> Database size </span></b>

### Sub - bloc

In [9]:
# Size constant
SIZES = {
    "integer": 8,
    "number": 8,        # Price, VAT
    "string": 80,       # Standard string
    "date": 20,         # Specific format
    "long_string": 200, # Desc, URL, Comment, Address
    "overhead": 12      # Key+Value overhead
}

# Stats (given by the lab)
GLOBAL_STATS = {
    # Global Stats
    "nb_clients": 10**7,
    "nb_products": 10**5,
    "nb_orderlines": 4*10**9,   
    "nb_warehouses": 200,       
    "nb_brands": 5000,
    "avg_cat_per_prod": 2       # on average 2 categories
}
# Derived Stats
DERIVED_STATS = { 
    # DB2 -> If stocks are embedded in product, how many?
    # Every product requires a stored stock entry for all 200 warehouses, even if the quantity is zero
    # So stocks_per_product = nb_warehouses
    "stocks_per_product" : 200,

    # DB5 -> If OrderLines are embedded in product (DB5), how many?
    # Uniform distribution hypothesis: Total Lines / Total Products
    "ol_per_product": int(GLOBAL_STATS["nb_orderlines"] / GLOBAL_STATS["nb_products"]), # 40,000
}

# Lists to identify specific field types
LONG_STRINGS = ["description", "image_url", "comment", "address", "headOffice"]
DATES = ["date", "birthDate", "deliveryDate"]

In [10]:
def get_field_size(field_name, field_type):
    """Calculates the size of a primitive field."""
    if field_type in ["integer", "number"]:
        return SIZES["integer"]
    elif field_type == "string":
        if field_name in DATES:
            return SIZES["date"]
        elif field_name in LONG_STRINGS:
            return SIZES["long_string"]
        else:
            return SIZES["string"]
    return 0

def calculate_doc_size(schema, context_name=""):
    """
    Recursively calculates the size of a document in bytes.
    """
    total_size = 0
    
    # CASE 1: Type = Object ==> Dictionary --> Recursiv call on all the properties fields
    if schema.get("type") == "object":
        properties = schema.get("properties", {})
        for key, sub_schema in properties.items():
            # Size = Key Overhead (12B) + Content Size
            field_size = calculate_doc_size(sub_schema, context_name=key)
            total_size += SIZES["overhead"] + field_size
            
    # CASE 2: Type = Array 
    elif schema.get("type") == "array":

        # Recursiv call on the items to find the size of 1 item
        item_schema = schema.get("items")
        single_item_size = calculate_doc_size(item_schema, context_name)
        
        # We have found the size of 1 item. How many items do we have?
        # Find an approximative size/lenght of the array using global statistics
        multiplier = 1
        
        # DB1/DB2/DB3/DB4/DB5: Categories embedded in Product
        if "categories" in context_name:
            multiplier = GLOBAL_STATS["avg_cat_per_prod"]
            
        # DB2: Stocks embedded in Product
        elif "stocks" in context_name:
            multiplier = DERIVED_STATS["stocks_per_product"]
            
        # DB5: OrderLines embedded in Product
        elif "order_lines" in context_name:
            multiplier = DERIVED_STATS["ol_per_product"]
        
        # Size = Array Overhead (12B) + (Nb items * Unit Size)
        total_size = SIZES["overhead"] + (multiplier * single_item_size)
        
    # CASE 3: Simple mapping -- type = integer, date, etc...
    else:
        field_type = schema.get("type")
        total_size = get_field_size(context_name, field_type)
        
    return total_size

def convert_bytes_to_gb(doc_size_bytes, collection_count):
    """Converts Bytes -> GB"""
    total_bytes = doc_size_bytes * collection_count
    return total_bytes / (1024**3)

def generate_report(db_name, schemas):
    print("-"*6 + f" SIZE REPORT {db_name} ---" + "-"*6 )
    total_db_size = 0
    
    for coll_name, schema in schemas.items():
        # 1. Size of ONE average document
        doc_size = calculate_doc_size(schema)
        
        # 2. Number of documents in the collection
        count = 0
        
        if coll_name == "Product":
            count = GLOBAL_STATS["nb_products"]
            
        elif coll_name == "Client":
            count = GLOBAL_STATS["nb_clients"]
            
        elif coll_name == "Warehouse":
            count = GLOBAL_STATS["nb_warehouses"]
            
        elif coll_name == "OrderLine":
            # DB1, DB2, DB3: Classic OrderLine Collection
            count = GLOBAL_STATS["nb_orderlines"]
            # DB4: OrderLine contains Product -> The count remains 4 Billion
            # (But unit doc_size will be huge)
            
        elif coll_name == "Stock":
            # DB1, DB4, DB5: Stock is an association table (Prod x Warehouse)
            count = GLOBAL_STATS["nb_products"] * GLOBAL_STATS["nb_warehouses"]
            # DB3: Stock contains Product -> The count remains Prod x Warehouse
        
        # 3. Final Calculation
        coll_size_gb = convert_bytes_to_gb(doc_size, count)
        total_db_size += coll_size_gb
        
        # Formatted Output
        print(f"Collection: {coll_name:<12} | Doc Size: {doc_size:>6.0f} B | Count: {count:>12,.0f} | Total: {coll_size_gb:>9.2f} GB")
        
    print(">" * 7 + f" TOTAL {db_name} : {total_db_size:.2f} GB\n")

# --- Execution ---
generate_report("DB1", db1_schemas)
generate_report("DB2", db2_schemas)
generate_report("DB3", db3_schemas)


------ SIZE REPORT DB1 ---------
Collection: Product      | Doc Size:   1428 B | Count:      100,000 | Total:      0.13 GB
Collection: Stock        | Doc Size:    152 B | Count:   20,000,000 | Total:      2.83 GB
Collection: Warehouse    | Doc Size:    252 B | Count:          200 | Total:      0.00 GB
Collection: OrderLine    | Doc Size:    356 B | Count: 4,000,000,000 | Total:   1326.20 GB
Collection: Client       | Doc Size:    632 B | Count:   10,000,000 | Total:      5.89 GB
>>>>>>> TOTAL DB1 : 1335.05 GB

------ SIZE REPORT DB2 ---------
Collection: Product      | Doc Size:  31852 B | Count:      100,000 | Total:      2.97 GB
Collection: Warehouse    | Doc Size:    252 B | Count:          200 | Total:      0.00 GB
Collection: OrderLine    | Doc Size:    356 B | Count: 4,000,000,000 | Total:   1326.20 GB
Collection: Client       | Doc Size:    632 B | Count:   10,000,000 | Total:      5.89 GB
>>>>>>> TOTAL DB2 : 1335.06 GB

------ SIZE REPORT DB3 ---------
Collection: Stock        

In [11]:
generate_report("DB4", db4_schemas)
generate_report("DB5", db5_schemas)

------ SIZE REPORT DB4 ---------
Collection: OrderLine    | Doc Size:   1776 B | Count: 4,000,000,000 | Total:   6616.12 GB
Collection: Stock        | Doc Size:    152 B | Count:   20,000,000 | Total:      2.83 GB
Collection: Warehouse    | Doc Size:    252 B | Count:          200 | Total:      0.00 GB
Collection: Client       | Doc Size:    632 B | Count:   10,000,000 | Total:      5.89 GB
>>>>>>> TOTAL DB4 : 6624.83 GB

------ SIZE REPORT DB5 ---------
Collection: Product      | Doc Size: 14241452 B | Count:      100,000 | Total:   1326.34 GB
Collection: Stock        | Doc Size:    152 B | Count:   20,000,000 | Total:      2.83 GB
Collection: Warehouse    | Doc Size:    252 B | Count:          200 | Total:      0.00 GB
Collection: Client       | Doc Size:    632 B | Count:   10,000,000 | Total:      5.89 GB
>>>>>>> TOTAL DB5 : 1335.06 GB



### What are the problems related to those denormalizations?

Based on the calculated database sizes, we observe critical issues with specific denormalization strategies. These results highlight the trade-offs between read performance and storage/maintenance costs.

#### 1. Extreme Data Redundancy (The DB4 Case)
**Observation**: DB4 is the heaviest database by far, reaching ~6.6 TB compared to ~1.3 TB for the normalized version.$\newline$
**The Problem**: By embedding the full Product object inside the OrderLine, we duplicate static and heavy data (Description, Image URL, Brand) 4.10**9 times (once for every order line).$\newline$
**Consequence**: This results in massive storage waste and financial inefficiency. It is an anti-pattern for large-scale datasets.$\newline$

#### 2. Unbounded Arrays & Technical Limits (The DB5 Case)
**Observation**: In the DB5, at each sell, an Order Line is added in the product that has been sold. Through the year, the number of order line for each products grows. This not a big issue for unpopular product but for the most famous one, the collections size while increase by number of order line time the size of an order line which is quite heavy$\newline$

**The Problem**: Embedding OrderLines inside Products creates "Unbounded Arrays" (Arrays with no size limit). Popular products (like Apple devices) will accumulate tens of thousands of order lines.$\newline$

**Consequence:** $\newline$
**Technical Failure**: MongoDB has a hard limit of 16 MB per document. A product with slightly more orders than average will fail to save. $\newline$
**Performance:** Reading a product just to display its price requires loading megabytes of useless order history into RAM (Over-fetching). $\newline$

#### 3. Update Anomalies (The DB3 Case)
**Observation**: The Stock collection grows from 2.8 GB (DB1) to 29 GB (DB3).
**The Problem**: By embedding Product details into Stock, the product information is duplicated 200 times (once per warehouse).$\newline$
**Consequence**: High maintenance cost. If a product description or price changes, the system must update 200 documents instead of just one, risking data inconsistency.$\newline$

#### Conclusion $\newline$
DB1 (Normalized) and DB2 (Product + Embedded Stocks) are the only viable solutions.$\newline$
DB2 is particularly efficient: it keeps the total volume low (only +3 GB vs DB1) while grouping Stock data with the Product to avoid joins during common queries.$\newline$
DB4 and DB5 are strictly non-viable due to storage costs and technical limitations.

<a id="libraries"></a>
# <b><span style='color:#fcc36d'>2.6 |</span><span style='color:#ff6200'> Sharding Strategies </span></b>

For each sharding strategy below, give the average number of documents per server & the average number of distinct values (for the sharding key) per server:

Average Documents per Server:$$\frac{\text{Total Documents in Collection}}{\text{Number of Servers}}$$Average Distinct Keys per Server:$$\frac{\text{Total Distinct Values of the Key}}{\text{Number of Servers}}$$

In [12]:
# CONFIG
NB_SERVERS = 1000
# For each products there is the quantity (which can be null) 
NB_STOCKS = GLOBAL_STATS["nb_products"]*GLOBAL_STATS["nb_warehouses"]

def analyze_sharding(strategy_name, collection_count, key_cardinality):
    """
    Computes averages for a given sharding strategy.
    """
    avg_docs = collection_count / NB_SERVERS
    avg_keys = key_cardinality / NB_SERVERS
    
    print(f"Strategy: {strategy_name}")
    print(f"  - Avg Docs/Server: {avg_docs:,.0f}")
    print(f"  - Avg Keys/Server: {avg_keys:,.2f}")
    print("-" * 40)

# --- 2.6.1 CALCULATIONS ---

# 1. St - #IDP
# Key = IDP (100,000 distinct values)
analyze_sharding("St - #IDP", NB_STOCKS, GLOBAL_STATS["nb_products"])

# 2. St - #IDW
# Key = IDW (200 distinct values)
analyze_sharding("St - #IDW", NB_STOCKS, GLOBAL_STATS["nb_warehouses"])

# 3. OL - #IDC
# Key = IDC (10,000,000 distinct values)
analyze_sharding("OL - #IDC", GLOBAL_STATS["nb_orderlines"], GLOBAL_STATS["nb_clients"])

# 4. OL - #IDP
# Key = IDP (100,000 distinct values)
analyze_sharding("OL - #IDP", GLOBAL_STATS["nb_orderlines"], GLOBAL_STATS["nb_products"])

# 5. Prod - #IDP
# Key = IDP (100,000 distinct values)
analyze_sharding("Prod - #IDP", GLOBAL_STATS["nb_products"], GLOBAL_STATS["nb_products"])

# 6. Prod - #brand
# Key = Brand (5,000 distinct values)
analyze_sharding("Prod - #brand", GLOBAL_STATS["nb_products"], GLOBAL_STATS["nb_brands"])

Strategy: St - #IDP
  - Avg Docs/Server: 20,000
  - Avg Keys/Server: 100.00
----------------------------------------
Strategy: St - #IDW
  - Avg Docs/Server: 20,000
  - Avg Keys/Server: 0.20
----------------------------------------
Strategy: OL - #IDC
  - Avg Docs/Server: 4,000,000
  - Avg Keys/Server: 10,000.00
----------------------------------------
Strategy: OL - #IDP
  - Avg Docs/Server: 4,000,000
  - Avg Keys/Server: 100.00
----------------------------------------
Strategy: Prod - #IDP
  - Avg Docs/Server: 100
  - Avg Keys/Server: 100.00
----------------------------------------
Strategy: Prod - #brand
  - Avg Docs/Server: 100
  - Avg Keys/Server: 5.00
----------------------------------------
